In [ ]:
import pandas as pd
import io

In [ ]:
def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
    sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
    sandbox_log =  sections[0].strip()
    activities_log = sections[1].split('Trade History:')[0]
    # sandbox_log_list = [json.loads(line) for line in sandbox_log.split('\n')]
    trade_history =  json.loads(sections[1].split('Trade History:')[1])
    # sandbox_log_df = pd.DataFrame(sandbox_log_list)
    market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_history_df = pd.json_normalize(trade_history)
    return market_data_df, trade_history_df

i = 1
df_24, _ = _process_data_(f'2024_data_logs/round_{i}.log')
df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

j = 2
df_23 = pd.read_csv(f"2023_data_logs/r{j}.csv", sep=';')

df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]

df_24.columns  = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]

df_test = df_24.merge(df_23, on='timestamp', how='inner')

In [ ]:
df_test.columns

In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

def get_centered_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_centered_with_{its}_its"] = (df[future_col] - df[prev_col])/df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    df.drop(columns=[future_col], inplace=True)
    return df

In [ ]:
import os
import json
import io
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats
from tqdm import tqdm

def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
        sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
        sandbox_log = sections[0].strip()
        activities_log = sections[1].split('Trade History:')[0]
        trade_history = json.loads(sections[1].split('Trade History:')[1])
        market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
        trade_history_df = pd.json_normalize(trade_history)
        return market_data_df, trade_history_df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

predictor_timeframes = [20]
responder_timeframes = [20]

results = []

for i in tqdm(range(1, 4)):
    for j in range(-1, 2):
        df_24 = pd.read_csv(f'2024_data_bottles/round-4-island-data-bottle/prices_round_4_day_{i}.csv', sep=';')
        df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-2/prices_round_2_day_{j}.csv", sep=';')
        df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
        df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
        
        df_test = df_24.merge(df_23, on='timestamp', how='inner')
        
        predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
        responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]
        
        df_copy = df_test.copy()
        
        for responder_timeframe in responder_timeframes:
            for symbol in responder_symbols:
                df_copy = get_future_returns(df_copy, symbol, responder_timeframe)
            
            for predictor_timeframe in predictor_timeframes:
                for symbol in predictor_symbols:
                    df_copy = get_future_returns(df_copy, symbol, predictor_timeframe)
                
                for predictor_symbol in predictor_symbols:
                    for responder_symbol in responder_symbols:
                        feature_col = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its") and col.startswith(predictor_symbol)]
                        target_col = f"{responder_symbol}_returns_in_{responder_timeframe}_its"
                        
                        df_train = df_copy[feature_col + [target_col]].dropna()
                        
                        X = df_train[feature_col]
                        y = df_train[target_col]
                        
                        model = LinearRegression(fit_intercept=False)
                        model.fit(X, y)
                        
                        y_pred = model.predict(X)
                        
                        r2 = r2_score(y, y_pred)
                        _, p_value = stats.pearsonr(y, y_pred)
                        
                        results.append({
                            '2024_day': i,
                            '2023_day': j,
                            'predictor_symbol': predictor_symbol,
                            'responder_symbol': responder_symbol,
                            'r_squared': r2,
                            'p_value': p_value,
                            'equation': f"{target_col} = {model.coef_[0]:.4f} * {feature_col[0]}"
                        })
                
                future_cols_predictor = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its")]
                df_copy.drop(columns=future_cols_predictor, inplace=True)
            
            future_cols_responder = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
            df_copy.drop(columns=future_cols_responder, inplace=True)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
results_df.head(20)

In [ ]:
import os
import json
import io
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats
from tqdm import tqdm

def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
        sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
        sandbox_log = sections[0].strip()
        activities_log = sections[1].split('Trade History:')[0]
        trade_history = json.loads(sections[1].split('Trade History:')[1])
        market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
        trade_history_df = pd.json_normalize(trade_history)
        return market_data_df, trade_history_df


predictor_timeframes = [1]
responder_timeframes = [1]

results = []

for i in tqdm(range(1, 4)):
    for j in range(-1, 2):
        df_24 = pd.read_csv(f'2024_data_bottles/round-4-island-data-bottle/prices_round_4_day_{i}.csv', sep=';')
        df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-2/prices_round_2_day_{j}.csv", sep=';')
        df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
        df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
        
        df_test = df_24.merge(df_23, on='timestamp', how='inner')
        
        predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
        responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]
        
        df_copy = df_test.copy()
        
        for responder_timeframe in responder_timeframes:
            for symbol in responder_symbols:
                df_copy = get_future_returns(df_copy, symbol, responder_timeframe)
            
            for predictor_timeframe in predictor_timeframes:
                for symbol in predictor_symbols:
                    df_copy = get_future_returns(df_copy, symbol, predictor_timeframe)
                
                for predictor_symbol in predictor_symbols:
                    for responder_symbol in responder_symbols:
                        feature_col = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its") and col.startswith(predictor_symbol)]
                        target_col = f"{responder_symbol}_returns_in_{responder_timeframe}_its"
                        
                        df_train = df_copy[feature_col + [target_col]].dropna()
                        
                        X = df_train[feature_col]
                        y = df_train[target_col]
                        
                        model = LinearRegression(fit_intercept=False)
                        model.fit(X, y)
                        
                        y_pred = model.predict(X)
                        
                        r2 = r2_score(y, y_pred)
                        _, p_value = stats.pearsonr(y, y_pred)
                        
                        results.append({
                            '2024_day': i,
                            '2023_day': j,
                            'predictor_symbol': predictor_symbol,
                            'responder_symbol': responder_symbol,
                            'r_squared': r2,
                            'p_value': p_value,
                            'equation': f"{target_col} = {model.coef_[0]:.4f} * {feature_col[0]}"
                        })
                
                future_cols_predictor = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its")]
                df_copy.drop(columns=future_cols_predictor, inplace=True)
            
            future_cols_responder = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
            df_copy.drop(columns=future_cols_responder, inplace=True)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
import os
import json
import io
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats
from tqdm import tqdm

def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
        sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
        sandbox_log = sections[0].strip()
        activities_log = sections[1].split('Trade History:')[0]
        trade_history = json.loads(sections[1].split('Trade History:')[1])
        market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
        trade_history_df = pd.json_normalize(trade_history)
        return market_data_df, trade_history_df

results = []

for i in tqdm(range(1, 4)):
    for j in range(-1, 2):
        df_24 = pd.read_csv(f'2024_data_bottles/round-4-island-data-bottle/prices_round_4_day_{i}.csv', sep=';')
        df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-2/prices_round_2_day_{j}.csv", sep=';')
        df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
        df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
        
        df_test = df_24.merge(df_23, on='timestamp', how='inner')
        
        predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
        responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]
        
        for predictor_symbol in predictor_symbols:
            for responder_symbol in responder_symbols:
                X = df_test[[predictor_symbol]]
                y = df_test[responder_symbol]
                
                for fit_intercept in [True, False]:
                    model = LinearRegression(fit_intercept=fit_intercept)
                    model.fit(X, y)
                    
                    y_pred = model.predict(X)
                    
                    r2 = r2_score(y, y_pred)
                    _, p_value = stats.pearsonr(y, y_pred)
                    
                    if fit_intercept:
                        equation = f"{responder_symbol} = {model.intercept_:.4f} + {model.coef_[0]:.4f} * {predictor_symbol}"
                    else:
                        equation = f"{responder_symbol} = {model.coef_[0]:.4f} * {predictor_symbol}"
                    
                    results.append({
                        '2024_day': i,
                        '2023_day': j,
                        'predictor_symbol': predictor_symbol,
                        'responder_symbol': responder_symbol,
                        'fit_intercept': fit_intercept,
                        'r_squared': r2,
                        'p_value': p_value,
                        'equation': equation
                    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

display(results_df.head(20))

In [ ]:
results_df.head(20)

In [ ]:
df_24 = pd.read_csv(f'2024_data_bottles/round-4-island-data-bottle/prices_round_4_day_{1}.csv', sep=';')
df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-2/prices_round_2_day_{-1}.csv", sep=';')
df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

In [ ]:
df_24 = df_24.rename(columns={'COCONUT': "COCONUT_CURR"})

In [ ]:
df_23 = df_23.rename(columns={'COCONUTS': "COCONUT_PREV"})

In [ ]:
df_23

In [ ]:
df = df_24[['timestamp', 'COCONUT_CURR']].merge(df_23[['timestamp', 'COCONUT_PREV']], on='timestamp', how='inner')

In [ ]:
df = get_future_returns(df, 'COCONUT_PREV', 50)
df = get_future_returns(df, 'COCONUT_CURR', 50)

In [ ]:
df_r2 = df.dropna()
r2_score(df_r2['COCONUT_PREV_returns_in_50_its'], df_r2['COCONUT_CURR_returns_in_50_its'])

In [ ]:
df['COCONUT_mult'] = 1 + df['COCONUT_PREV_returns_in_1_its']

In [ ]:
df['COCONUT_pred'] = df['COCONUT_CURR']*df['COCONUT_mult']

In [ ]:
df["COCONUT_pred"] = df["COCONUT_pred"].round(0) + (df["COCONUT_pred"] % 1 >= 0.5) * 0.5

In [ ]:
df['COCONUT_pred'] = df['COCONUT_pred'].shift(1)

In [ ]:
df_r2 = df.dropna()
r2_score(df_r2['COCONUT_CURR'], df_r2['COCONUT_pred'])

In [ ]:
df.columns

In [ ]:
import plotly.graph_objects as go

# Create traces for each return series
trace_curr_1 = go.Scatter(x=df['timestamp'], y=df['COCONUT_CURR_returns_in_1_its'],
                           mode='lines', name='Current Returns (1 iteration)')
trace_prev_1 = go.Scatter(x=df['timestamp'], y=df['COCONUT_PREV_returns_in_1_its'],
                           mode='lines', name='Previous Returns (1 iteration)')

trace_curr_20 = go.Scatter(x=df['timestamp'], y=df['COCONUT_CURR_returns_in_20_its'],
                            mode='lines', name='Current Returns (20 iterations)')
trace_prev_20 = go.Scatter(x=df['timestamp'], y=df['COCONUT_PREV_returns_in_20_its'],
                            mode='lines', name='Previous Returns (20 iterations)')

trace_curr_50 = go.Scatter(x=df['timestamp'], y=df['COCONUT_CURR_returns_in_50_its'],
                            mode='lines', name='Current Returns (50 iterations)')
trace_prev_50 = go.Scatter(x=df['timestamp'], y=df['COCONUT_PREV_returns_in_50_its'],
                            mode='lines', name='Previous Returns (50 iterations)')

# Create the figure and add the traces
fig = go.Figure()
fig.add_trace(trace_curr_1)
fig.add_trace(trace_prev_1)
fig.add_trace(trace_curr_20)
fig.add_trace(trace_prev_20)
fig.add_trace(trace_curr_50)
fig.add_trace(trace_prev_50)

# Set the layout
fig.update_layout(
    title='Returns Over Iterations',
    xaxis_title='Timestamp',
    yaxis_title='Returns',
    legend_title='Return Series'
)

# Display the plot
fig.show()

In [ ]:
# Initialize the first value of "COCONUT_stupid" as 8000
df.loc[0, 'COCONUT_stupid'] = 10000

# Calculate the subsequent values of "COCONUT_stupid" based on the previous row's values
for i in range(1, len(df)):
    df.loc[i, 'COCONUT_stupid'] = round(2*(df.loc[i-1, 'COCONUT_stupid'] * df.loc[i-1, 'COCONUT_mult']))/2
    

In [ ]:
df

In [ ]:
df_r2 = df.copy()
df_r2 = df_r2.dropna()
r2_score(df_r2['COCONUT_CURR'], df_r2['COCONUT_stupid'])

In [ ]:
for i in tqdm(range(1, 4)):
    j = i - 2
    df_24 = pd.read_csv(f'2024_data_bottles/round-4-island-data-bottle/prices_round_4_day_{i}.csv', sep=';')
    df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

    df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-2/prices_round_2_day_{j}.csv", sep=';')
    df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
    df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
    df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]

    df_test = df_24.merge(df_23, on='timestamp', how='inner')
    display(df_test[['timestamp','COCONUT_curr', 'COCONUTS_past']])

# price change space

In [ ]:
def get_future_price_change(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_price_change_in_{its}_its"] = df[future_col] - df[col]
    df.drop(columns=[future_col], inplace=True)
    return df


predictor_timeframes = [1]
responder_timeframes = [1]

results = []

for i in tqdm(range(1, 4)):
    for j in range(-1, 2):
        df_24 = pd.read_csv(f'2024_data_bottles/round-4-island-data-bottle/prices_round_4_day_{i}.csv', sep=';')
        df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-2/prices_round_2_day_{j}.csv", sep=';')
        df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
        df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
        
        df_test = df_24.merge(df_23, on='timestamp', how='inner')
        
        predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
        responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]
        
        df_copy = df_test.copy()
        
        for responder_timeframe in responder_timeframes:
            for symbol in responder_symbols:
                df_copy = get_future_price_change(df_copy, symbol, responder_timeframe)
            
            for predictor_timeframe in predictor_timeframes:
                for symbol in predictor_symbols:
                    df_copy = get_future_price_change(df_copy, symbol, predictor_timeframe)
                
                for predictor_symbol in predictor_symbols:
                    for responder_symbol in responder_symbols:
                        feature_col = [col for col in df_copy.columns if col.endswith(f"_price_change_in_{predictor_timeframe}_its") and col.startswith(predictor_symbol)]
                        target_col = f"{responder_symbol}_price_change_in_{responder_timeframe}_its"
                        
                        df_train = df_copy[feature_col + [target_col]].dropna()
                        
                        X = df_train[feature_col]
                        y = df_train[target_col]
                        
                        model = LinearRegression(fit_intercept=False)
                        model.fit(X, y)
                        
                        y_pred = model.predict(X)
                        
                        r2 = r2_score(y, y_pred)
                        _, p_value = stats.pearsonr(y, y_pred)
                        
                        results.append({
                            '2024_day': i,
                            '2023_day': j,
                            'predictor_symbol': predictor_symbol,
                            'responder_symbol': responder_symbol,
                            'r_squared': r2,
                            'p_value': p_value,
                            'equation': f"{target_col} = {model.coef_[0]:.4f} * {feature_col[0]}"
                        })
                
                future_cols_predictor = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its")]
                df_copy.drop(columns=future_cols_predictor, inplace=True)
            
            future_cols_responder = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
            df_copy.drop(columns=future_cols_responder, inplace=True)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
display(results_df.head(20))

In [ ]:
df

# roses

In [ ]:
for i in tqdm(range(0, 3)):
    j = i
    df_24 = pd.read_csv(f'2024_data_bottles/round-3-island-data-bottle/prices_round_3_day_{i}.csv', sep=';')
    df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

    df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-3/prices_round_3_day_{j}.csv", sep=';')
    df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
    df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
    df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]

    df_test = df_24.merge(df_23, on='timestamp', how='inner')
    print(df_test.columns)
    display(df_test[['timestamp','ROSES_curr', 'DIVING_GEAR_past']])

In [ ]:
(98904.0-98915.0)/98915.0

In [ ]:
(14545.5-14550.5)/14550.5

In [ ]:
-0.00011120659151796997*3.07

# ugh cross day over multiple augh 

In [ ]:
import os
import json
import io
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats
from tqdm import tqdm

def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
        sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
        sandbox_log = sections[0].strip()
        activities_log = sections[1].split('Trade History:')[0]
        trade_history = json.loads(sections[1].split('Trade History:')[1])
        market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
        trade_history_df = pd.json_normalize(trade_history)
        return market_data_df, trade_history_df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

predictor_timeframes = [20]
responder_timeframes = [20]

results = []

for i in tqdm(range(1, 5)):  # Iterate over rounds 1 to 5 for 2024 data
    if i == 2:
        continue
    for j in range(1, 5):  # Iterate over rounds 1 to 5 for 2023 data
        for k in range(i - 3, i):  # Iterate over CSV files indexed from round_number - 3 to round_number - 1
            df_24 = pd.read_csv(f'2024_data_bottles/round-{i}-island-data-bottle/prices_round_{i}_day_{k}.csv', sep=';')
            df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
            df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
            for l in range(j - 3, j):  # Iterate over CSV files indexed from round_number - 3 to round_number - 1

                df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-{j}/prices_round_{j}_day_{l}.csv", sep=';')
                df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

                df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]

                df_test = df_24.merge(df_23, on='timestamp', how='inner')

                predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
                responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]

                df_copy = df_test.copy()
        
        for responder_timeframe in responder_timeframes:
            for symbol in responder_symbols:
                df_copy = get_future_returns(df_copy, symbol, responder_timeframe)
            
            for predictor_timeframe in predictor_timeframes:
                for symbol in predictor_symbols:
                    df_copy = get_future_returns(df_copy, symbol, predictor_timeframe)
                
                for predictor_symbol in predictor_symbols:
                    for responder_symbol in responder_symbols:
                        feature_col = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its") and col.startswith(predictor_symbol)]
                        target_col = f"{responder_symbol}_returns_in_{responder_timeframe}_its"
                        
                        df_train = df_copy[feature_col + [target_col]].dropna()
                        
                        X = df_train[feature_col]
                        y = df_train[target_col]
                        
                        model = LinearRegression(fit_intercept=False)
                        model.fit(X, y)
                        
                        y_pred = model.predict(X)
                        
                        r2 = r2_score(y, y_pred)
                        _, p_value = stats.pearsonr(y, y_pred)
                        
                        results.append({
                            '2024_day': i,
                            '2023_day': j,
                            'predictor_symbol': predictor_symbol,
                            'responder_symbol': responder_symbol,
                            'r_squared': r2,
                            'p_value': p_value,
                            'equation': f"{target_col} = {model.coef_[0]:.4f} * {feature_col[0]}"
                        })
                
                future_cols_predictor = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its")]
                df_copy.drop(columns=future_cols_predictor, inplace=True)
            
            future_cols_responder = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
            df_copy.drop(columns=future_cols_responder, inplace=True)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
results_df.head(50)

# roses basket

In [ ]:
def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

predictor_timeframes = [50]
responder_timeframes = [50]

results = []

for i in tqdm(range(0, 3)):
    df_24 = pd.read_csv(f'2024_data_bottles/round-3-island-data-bottle/prices_round_3_day_{i}.csv', sep=';')
    df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
    
    df_23 = pd.read_csv(f"2023_data_bottles/island-data-bottle-round-3/prices_round_3_day_{i}.csv", sep=';')
    df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
    
    df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
    df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
    
    df_test = df_24.merge(df_23, on='timestamp', how='inner')
    
    predictor_symbol = 'DIVING_GEAR_past'
    responder_symbol = 'GIFT_BASKET_curr'
    
    df_copy = df_test.copy()
    df_copy['ROSES_IMPLIED_curr'] = df_copy['GIFT_BASKET_curr'] - 4*df_copy['CHOCOLATE_curr'] - 6*df_copy['STRAWBERRIES_curr']
     
    for responder_timeframe in responder_timeframes:
        df_copy = get_future_returns(df_copy, responder_symbol, responder_timeframe)
        
        for predictor_timeframe in predictor_timeframes:
            df_copy = get_future_returns(df_copy, predictor_symbol, predictor_timeframe)
            
            feature_col = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its") and col.startswith(predictor_symbol)]
            target_col = f"{responder_symbol}_returns_in_{responder_timeframe}_its"
            
            df_train = df_copy[feature_col + [target_col]].dropna()
            
            X = df_train[feature_col]
            y = df_train[target_col]
            
            model = LinearRegression(fit_intercept=False)
            model.fit(X, y)
            
            y_pred = model.predict(X)
            
            r2 = r2_score(y, y_pred)
            _, p_value = stats.pearsonr(y, y_pred)
            
            results.append({
                'day': i,
                'predictor_symbol': predictor_symbol,
                'responder_symbol': responder_symbol,
                'r_squared': r2,
                'p_value': p_value,
                'equation': f"{target_col} = {model.coef_[0]:.4f} * {feature_col[0]}"
            })
            
            future_cols_predictor = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its")]
            df_copy.drop(columns=future_cols_predictor, inplace=True)
        
        future_cols_responder = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
        df_copy.drop(columns=future_cols_responder, inplace=True)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
results_df